Import All Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

ModuleNotFoundError: No module named 'pandas'

# Load and Explore Raw Data

In [ ]:

# Load the raw Uber data
df = pd.read_csv('../data/raw/cab_rides_150k.csv')
# Display the first few rows
print("First 5 rows of the dataset:")
print(df.head())

# Check data types and missing values
print("\nData types and non-null counts:")
print(df.info())

# Basic statistics
print("\nBasic statistics:")
print(df.describe())

# Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())
df.shape

First 5 rows of the dataset:
   distance cab_type     time_stamp       destination  \
0      1.89     Uber  1545085210334  Theatre District   
1      1.97     Lyft  1543366102245  Theatre District   
2      1.35     Uber  1543701476008         North End   
3      0.91     Uber  1544793611629       Beacon Hill   
4      1.23     Lyft  1544851211219          West End   

                    source  price  surge_multiplier  \
0  Northeastern University    8.0               1.0   
1  Northeastern University   13.5               1.0   
2              Beacon Hill    NaN               1.0   
3         Haymarket Square    NaN               1.0   
4                North End   13.5               1.0   

                                     id                            product_id  \
0  ad4dc1c8-74e4-4018-aefe-be4e4b8e1ee5  55c66225-fbe7-4fd5-9072-eab1ece5e23e   
1  47547461-f88b-4c14-9920-1d4a79e7025f                             lyft_plus   
2  45fba16d-75f1-4e10-bbb7-614f7f1624ae  8cf7e821-f0d3

(150000, 10)

In [5]:
df1 =pd.read_csv('../data/raw/cab_rides.csv')
# Display first rows
print("First 5 rows of df1:")
print(df1.head())

# Data types and non-null values
print("\nData types and non-null counts (df1):")
print(df1.info())

# Basic statistics
print("\nBasic statistics (df1):")
print(df1.describe())

# Missing values
print("\nMissing values per column (df1):")
print(df1.isnull().sum())
df1.shape

(693071, 10)

In [6]:
df2=pd.read_csv('../data/raw/weather.csv')
# Display first rows
print("First 5 rows of df2:")
print(df2.head())

# Data types and non-null values
print("\nData types and non-null counts (df2):")
print(df2.info())

# Basic statistics
print("\nBasic statistics (df2):")
print(df2.describe())

# Missing values
print("\nMissing values per column (df2):")
print(df2.isnull().sum())df2.shape

(6276, 8)

# Data Cleaning and Preprocessing

In [14]:
df_clean=df.dropna()
print(df_clean)

df1_clean=df1.dropna()
print(df1_clean)

df2_clean=df2.dropna()
print(df2_clean)

        distance cab_type     time_stamp         destination  \
0           1.89     Uber  1545085210334    Theatre District   
1           1.97     Lyft  1543366102245    Theatre District   
4           1.23     Lyft  1544851211219            West End   
5           4.28     Lyft  1543575478661  Financial District   
6           2.34     Uber  1545066909520            Back Bay   
...          ...      ...            ...                 ...   
149995      1.21     Lyft  1545116710840  Financial District   
149996      2.31     Lyft  1543865877691       North Station   
149997      0.56     Lyft  1545044114463  Financial District   
149998      2.83     Lyft  1543775577675            West End   
149999      3.44     Lyft  1543640583161       North Station   

                         source  price  surge_multiplier  \
0       Northeastern University    8.0               1.0   
1       Northeastern University   13.5               1.0   
4                     North End   13.5             

In [19]:
# Data Cleaning and Preprocessing
# Handle missing values (drop rows with missing values for simplicity)
# Normalize fare column name
fare_col = 'fare_amount' if 'fare_amount' in df_clean.columns else 'price'
if fare_col not in df_clean.columns:
    raise ValueError("Expected fare column 'fare_amount' or 'price' in the input data.")

# Remove outliers (e.g., fares <= 0 or > 100, distances unreasonable)
df_clean = df_clean[(df_clean[fare_col] > 0) & (df_clean[fare_col] < 100)]

# Normalize datetime columns for this dataset
if 'pickup_datetime' in df_clean.columns:
    df_clean['pickup_datetime'] = pd.to_datetime(df_clean['pickup_datetime'])
elif 'time_stamp' in df_clean.columns:
    df_clean['pickup_datetime'] = pd.to_datetime(df_clean['time_stamp'], unit='ms', errors='coerce')
else:
    raise KeyError("Expected timestamp in 'pickup_datetime' or 'time_stamp'.")

if 'dropoff_datetime' in df_clean.columns:
    df_clean['dropoff_datetime'] = pd.to_datetime(df_clean['dropoff_datetime'])

# Save the cleaned data
df_clean.to_csv('../data/processed/clean_data.csv', index=False)
print("Cleaned data saved to '../data/processed/clean_data.csv'")

Cleaned data saved to '../data/processed/clean_data.csv'


In [20]:
df_clean.shape

(138135, 11)

# Feature Engineering

In [ ]:
# Feature Engineering
import numpy as np

# Calculate distance in a robust way
if {'pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude'}.issubset(df_clean.columns):
    def haversine_distance(lat1, lon1, lat2, lon2):
        R = 6371  # Earth radius in km
        dlat = np.radians(lat2 - lat1)
        dlon = np.radians(lon2 - lon1)
        a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
        c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
        return R * c

    df_clean['distance_km'] = haversine_distance(df_clean['pickup_latitude'], df_clean['pickup_longitude'], 
                                                df_clean['dropoff_latitude'], df_clean['dropoff_longitude'])

elif 'distance' in df_clean.columns:
    # Use existing distance field from dataset; assume it's in kilometers already
    df_clean['distance_km'] = df_clean['distance']

else:
    raise KeyError("Expected either pickup/dropoff latitude/longitude columns or 'distance' column")

# Time-based features
if 'pickup_datetime' not in df_clean.columns and 'time_stamp' in df_clean.columns:
    df_clean['pickup_datetime'] = pd.to_datetime(df_clean['time_stamp'], unit='ms', errors='coerce')

if 'pickup_datetime' in df_clean.columns:
    df_clean['pickup_hour'] = df_clean['pickup_datetime'].dt.hour
    df_clean['pickup_day'] = df_clean['pickup_datetime'].dt.dayofweek
    df_clean['pickup_month'] = df_clean['pickup_datetime'].dt.month
else:
    raise KeyError("Expected timestamp in 'pickup_datetime' or 'time_stamp' for feature engineering")

# Categorical encodings (if any, e.g., passenger_count as is, or one-hot if needed)
# For simplicity, keep numerical

print("New features added: distance_km, pickup_hour, pickup_day, pickup_month")

# Exploratory Data Analysis

In [ ]:
# Exploratory Data Analysis
import matplotlib.pyplot as plt
import seaborn as sns

# Histograms
df_clean.hist(figsize=(10, 8))
plt.show()

# Correlation heatmap
plt.figure(figsize=(8, 6))
num_df = df_clean.select_dtypes(include=['number'])
sns.heatmap(num_df.corr(), annot=True, cmap='coolwarm')
plt.show()

# Scatter plot: distance vs fare
fare_col = 'fare_amount' if 'fare_amount' in df_clean.columns else 'price'
plt.scatter(df_clean['distance_km'], df_clean[fare_col])
plt.xlabel('Distance (km)')
plt.ylabel('Fare Amount')
plt.title('Distance vs Fare Amount')
plt.show()

# Model Training

In [ ]:
# --- 1. Additional Imports ---
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import warnings
warnings.filterwarnings('ignore')

# --- 2. Prepare Features and Target ---
target_col = 'surge_multiplier'

feature_cols = ['distance_km', 'pickup_hour', 'pickup_day', 'pickup_month', 'price']

if 'cab_type' in df_clean.columns:
    df_clean['cab_type_encoded'] = df_clean['cab_type'].map({'Uber': 0, 'Lyft': 1}).fillna(-1).astype(int)
    feature_cols.append('cab_type_encoded')

if 'name' in df_clean.columns:
    from sklearn.preprocessing import LabelEncoder
    le_name = LabelEncoder()
    df_clean['name_encoded'] = le_name.fit_transform(df_clean['name'].astype(str))
    feature_cols.append('name_encoded')

X = df_clean[feature_cols].copy()
y = df_clean[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')
print(f'Features used: {feature_cols}')

# --- 3. Hyperparameter Tuning with RandomizedSearchCV ---
param_distributions = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

xgb_model = XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=12,
    scoring='r2',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

best_model = random_search.best_estimator_
print(f'\nBest Parameters: {random_search.best_params_}')
print(f'Best CV R² Score: {random_search.best_score_:.4f}')


# Model Evaluation

In [ ]:
# --- 5. Evaluation on Test Data ---
y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('=== Model Evaluation on Test Set ===')
print(f'R² Score : {r2:.4f}')
print(f'MAE      : {mae:.4f}')
print(f'RMSE     : {rmse:.4f}')

# --- 6. Custom Accuracy (within 10% tolerance) ---
tolerance = 0.10
within_tolerance = np.abs(y_pred - y_test.values) <= tolerance * np.abs(y_test.values)
custom_accuracy = np.mean(within_tolerance) * 100
print(f'\nModel Accuracy: {custom_accuracy:.2f}%')

# --- 7. Overfitting Check ---
y_train_pred = best_model.predict(X_train)
train_r2 = r2_score(y_train, y_train_pred)
print(f'\nTrain R² : {train_r2:.4f}')
print(f'Test  R² : {r2:.4f}')
print(f'Gap      : {abs(train_r2 - r2):.4f} (low gap = no overfitting)')


# Make Predictions

In [ ]:
# --- 8. Predict Surge Values and Display Sample Predictions ---
import pandas as pd

predictions_df = pd.DataFrame({
    'Actual Surge': y_test.values,
    'Predicted Surge': np.round(y_pred, 4),
    'Abs Error': np.round(np.abs(y_pred - y_test.values), 4)
})

print('=== Sample Predictions (first 20) ===')
print(predictions_df.head(20).to_string(index=False))


In [ ]:
# --- Final Model Summary ---
print('=' * 50)
print('       FINAL MODEL SUMMARY')
print('=' * 50)
print(f'Model          : XGBRegressor (Tuned)')
print(f'Best Params    : {random_search.best_params_}')
print(f'Features       : {feature_cols}')
print(f'Train Samples  : {X_train.shape[0]}')
print(f'Test Samples   : {X_test.shape[0]}')
print(f'R² Score       : {r2:.4f}')
print(f'MAE            : {mae:.4f}')
print(f'RMSE           : {rmse:.4f}')
print(f'Custom Accuracy: {custom_accuracy:.2f}%')
print('=' * 50)
